# Demo 10 - Command-line entropy (obfuscation hunting)

**Fast** (distinct commands, bounded) · **Pool:** Medium · **Visual:** histogram + bar

**The question:** which command lines look machine-generated or deliberately obscured?

Encoded and obfuscated commands use a far wider spread of characters than ordinary ones. We
measure that spread mathematically, combine it with how rarely each command was seen, and
rank what comes out.

There is no KQL function for this. It is an information-theory calculation, and a notebook
lets you define arbitrary maths per row and sort on the result.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `WORKSPACE` - which workspace to read `DeviceProcessEvents` from.
- `LOOKBACK_DAYS` - how much process history to score.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 7

## 3. Collect the distinct command lines and how often each ran

Group `DeviceProcessEvents` by the exact command line and count occurrences. Command lines
shorter than 20 characters are dropped: they are almost all routine, and too short for the
entropy score to mean anything.

`.orderBy("freq").limit(4000)` deliberately keeps the **least** common 4,000. Rare is where
the interesting things live. The command lines that run ten thousand times a day are your
normal, by definition.

In [ ]:
import pandas as pd

proc = data_provider.read_table("DeviceProcessEvents", WORKSPACE)
cmds = (proc.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
            .filter(F.col("ProcessCommandLine").isNotNull())
            .groupBy("ProcessCommandLine").agg(F.count("*").alias("freq"))
            .filter(F.length("ProcessCommandLine") > 20)
            .orderBy("freq").limit(4000)).toPandas()
print("distinct commands:", len(cmds))

## 4. Score each command line for randomness, then rank

**Shannon entropy** measures how unpredictable the characters in a string are, in bits per
character. It is the same idea that underpins password strength.

- Ordinary readable commands (`ping`, `net use \\server\share`) reuse a small set of
  letters and score low, roughly 3 to 4 bits per character.
- Base64 blobs, encrypted payloads and randomly generated filenames use the whole alphabet
  fairly evenly and score high, roughly 5 to 6 bits.

Entropy alone is not enough, because plenty of legitimate software passes long encoded
arguments every single day. So we combine it with **rarity** (one divided by how often the
command ran). High entropy *and* rarely seen is the combination worth investigating.

Left chart: the distribution of entropy across everything, with the 98th percentile marked.
Right chart: the twelve highest-scoring rare commands.

The labels are numbered because two different commands can share their first 38 characters,
and unnumbered duplicate labels would stack on the same tick and hide a bar.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def shannon(s):
    if not s: return 0.0
    _, counts = np.unique(list(s), return_counts=True)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())

if cmds.empty:
    print(f"No command lines over 20 chars in the last {LOOKBACK_DAYS} days - raise LOOKBACK_DAYS.")
else:
    cmds["entropy"] = cmds["ProcessCommandLine"].map(shannon)
    cmds["rarity"] = 1.0 / cmds["freq"]
    cmds["score"] = cmds["entropy"] * (0.5 + cmds["rarity"])
    thr = cmds["entropy"].quantile(0.98)
    suspicious = cmds.sort_values("score", ascending=False).head(12)

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].hist(cmds["entropy"], bins=40, color="#8e44ad", alpha=.8)
    ax[0].axvline(thr, ls="--", color="red", label=f"98th pct = {thr:.2f}")
    ax[0].set_title("Command-line entropy distribution"); ax[0].set_xlabel("bits/char"); ax[0].legend()

    # Number the labels: two commands sharing their first 38 chars would otherwise
    # collapse onto one categorical tick and hide a bar.
    labels = [f"{i+1}. {str(c)[:38]}" for i, c in enumerate(suspicious["ProcessCommandLine"])]
    ax[1].barh(labels[::-1], suspicious["entropy"].values[::-1], color="#c0392b")
    ax[1].set_title("Highest-entropy rare commands"); ax[1].set_xlabel("entropy")
    plt.tight_layout(); plt.show()

suspicious[["ProcessCommandLine","freq","entropy","score"]] if not cmds.empty else "no data"

## Why a notebook beats KQL here

Shannon entropy is `-sum(p*log2 p)` over the character distribution - an information-theory metric with no KQL equivalent. Notebooks let you define arbitrary maths per row and rank on it.